# multirotor-extended-1


## Per-rotor parameters

The first modification we need is to provide per-rotor parameters. For this extension we have the exact same parameters defining the aerodynamics and dynamics of the rotors, that doesn't change, we only allow for them to be different (the asymmetric objective).

| parameter | what it is |
|---|---|
| `k_eta` | thrust coefficient |
| `k_m` | yaw moment (reaction torque) coefficient |
| `k_d` | rotor drag, in-plane |
| `k_z` | induced inflow, axial |
| `k_h` | translational lift |
| `k_flap` | hub flapping moment coefficient |
| `rotor_thrust_axes` | thrust direction of each rotor in the body frame, `(num_rotors, 3)` |
| `tau_m` | motor response time |
| `rotor_speed_min` | minimum rotor speed |
| `rotor_speed_max` | maximum rotor speed |
| `rotor_inertia` | rotating inertia (prop + motor bell) about its own axis |

## Core physics changes


For the core physics, we need 2 key modifications: one for the allocation matrix, and another one
for the angular momentum terms and gyro effects, that are accounted now that the rotors axis may not
always be in the z direction, so a net angular momentum exists and would not be cancelled.


## Allocation matrix change

Generalizing the force directions and not assuming directions of thrust and reaction torque on
rotors in fixed body coordinates.

$$
\texttt{f\_to\_TM} =
\begin{bmatrix}
\mathbf 1^\top \\[2pt]
\big[\mathbf r_i \times \hat z\big]_{x,y} \\[2pt]
k\,\sigma_i
\end{bmatrix}
\qquad\text{where}\quad k = \frac{k_m}{k_\eta}
$$

$$
\mathbf m_i = \mathbf r_i \times \hat{\mathbf n}_i
\;+\; \sigma_i\,\frac{k_{m,i}}{k_{\eta,i}}\,\hat{\mathbf n}_i ,
\qquad
\texttt{f\_to\_TM} =
\begin{bmatrix}
\hat{\mathbf n}_i \cdot \hat z \\[2pt]
\mathbf m_i
\end{bmatrix}
$$


In [ ]:
# original
k = k_m/k_eta
f_to_TM = np.vstack((np.ones((1,num_rotors)),np.hstack([np.cross(rotor_pos[key],np.array([0,0,1])).reshape(-1,1)[0:2] for key in rotor_pos]), (k * rotor_dir).reshape(1,-1)))

# extended
rotor_moments = np.cross(rotor_geometry, rotor_thrust_axes)
rotor_moments += (rotor_dir * k_m / k_eta)[:, np.newaxis] * rotor_thrust_axes
f_to_TM = np.vstack((rotor_thrust_axes[:, 2], rotor_moments.T))


## Gyroscopic and angular momentum changes

We now calculate each rotor momentum axis as

$$
\texttt{rotor\_momentum\_axes}_i = \sigma_i\,J_i\,\hat{\mathbf n}_i,
\qquad
\mathbf h_\text{int} = -\sum_i \sigma_i\,J_i\,\Omega_i\,\hat{\mathbf n}_i
$$

so rotor $i$ spins about $-\sigma_i\hat{\mathbf n}_i$.

$\boldsymbol\omega \times \mathbf h_\text{int}$ — gyroscopic precession — is for the precession of
the potentially not cancelled angular momentum directions of the rotors.

$\dot{\mathbf h}_\text{int}$, albeit not necessarily a requirement of the 2 objectives, adds spin up
reactions.

$$
\dot{\boldsymbol\omega} = \mathbf I^{-1}\big(\mathbf M - \boldsymbol\omega \times \mathbf I\boldsymbol\omega\big)
$$

$$
\dot{\boldsymbol\omega} = \mathbf I^{-1}\Big(\mathbf M
- \boldsymbol\omega \times \big(\mathbf I\boldsymbol\omega + \mathbf h_\text{int}\big)
- \dot{\mathbf h}_\text{int}\Big)
$$


In [ ]:
# original
w_hat = hat_map(w)
w_dot = inv_inertia @ (MtotB - w_hat @ (inertia @ w))

# extended
rotor_momentum_axes = rotor_thrust_axes.T * (rotor_dir * rotor_inertia)

h_int = -rotor_momentum_axes @ rotor_speeds
h_int_dot = -rotor_momentum_axes @ rotor_accel
w_dot = inv_inertia @ (MtotB - np.cross(w, inertia @ w + h_int) - h_int_dot)


## Resolving forces on rotor's own spin axis

$$
\mathbf T_i = k_\eta\,\Omega_i^2\,\hat z
\qquad\longrightarrow\qquad
\mathbf T_i = k_{\eta,i}\,\Omega_i^2\,\hat{\mathbf n}_i
$$

$$
\boldsymbol\tau_i = \sigma_i\,k_m\,\Omega_i^2\,\hat z
\qquad\longrightarrow\qquad
\boldsymbol\tau_i = \sigma_i\,k_{m,i}\,\Omega_i^2\,\hat{\mathbf n}_i
$$

Previously `rotor_drag_matrix` assumed also that the direction of drag was in the z plane. Now it
isn't. Now we also use the already calculated local airspeed that the body sees, to get axial and
in-plane speeds according to each rotor's $\hat{\mathbf n}$, and get all forces on the proper
direction.

$$
\mathbf v_{\text{ax},i} = \hat{\mathbf n}_i\,\big(\hat{\mathbf n}_i \cdot \mathbf v_{\text{loc},i}\big),
\qquad
\mathbf v_{\text{ip},i} = \mathbf v_{\text{loc},i} - \mathbf v_{\text{ax},i}
$$


In [ ]:
local_airspeeds = body_airspeed_vector[:, np.newaxis] + hat_map(body_rates)@(rotor_geometry.T)

# original
rotor_drag_matrix = np.array([[k_d,   0,     0],
                              [0,     k_d,   0],
                              [0,     0,     k_z]])

T = np.array([0, 0, k_eta])[:, np.newaxis]*rotor_speeds**2
H = -rotor_speeds*(rotor_drag_matrix@local_airspeeds)
M_flap = -k_flap*rotor_speeds*((hat_map(local_airspeeds.T).transpose(2, 0, 1))@np.array([0,0,1])).T
T += np.array([0, 0, k_h])[:, np.newaxis]*(local_airspeeds[0, :]**2 + local_airspeeds[1, :]**2)
M_yaw = rotor_dir*(np.array([0, 0, k_m])[:, np.newaxis]*rotor_speeds**2)

# extended
T = rotor_thrust_axes.T * (k_eta * rotor_speeds**2)

axes = rotor_thrust_axes.T
axial_airspeeds = axes * np.sum(axes * local_airspeeds, axis=0)
inplane_airspeeds = local_airspeeds - axial_airspeeds

H = -rotor_speeds*(k_d*inplane_airspeeds + k_z*axial_airspeeds)
M_flap = -k_flap*rotor_speeds*np.cross(local_airspeeds.T, axes.T).T
T += axes*(k_h*np.sum(inplane_airspeeds**2, axis=0))
M_yaw = rotor_thrust_axes.T * (rotor_dir * k_m * rotor_speeds**2)
